In [1]:
# !pip install PyMuPDF
# !pip install pdfplumber

# Modules: 

In [2]:
import os
from typing import Any
import pandas as pd
import pdfplumber

try:
    import fitz  # PyMuPDF

    _HAS_FITZ = True
except Exception:
    _HAS_FITZ = False


def _extract_images_with_fitz(pdf_path: str, page_number: int, images_dir: str | None) -> list[dict[str, Any]]:
    """Extract images from a single page using PyMuPDF.

    Returns a list of dicts with keys: `bytes`, `ext`, `xref`, and optionally `path`.
    """
    results: list[dict[str, Any]] = []
    doc = fitz.open(pdf_path)
    doc_name = pdf_path.split("/")[-1].split(".")[0]
    try:
        page = doc[page_number - 1]
        imgs = page.get_images(full=True)
        for idx, img in enumerate(imgs):
            xref = img[0]
            base = doc.extract_image(xref)
            image_bytes = base.get("image")
            ext = base.get("ext", "png")
            info = {"ext": ext, "xref": xref, "index": idx}  # "bytes": image_bytes, 
            if images_dir:
                os.makedirs(images_dir, exist_ok=True)
                filename = f"{doc_name}_{page_number}_{idx}.{ext}"
                out_path = os.path.join(images_dir, filename)
                with open(out_path, "wb") as f:
                    # print(out_path)
                    f.write(image_bytes)
                info["path"] = out_path
            results.append(info)
    except Exception as e:
        print(e)
    finally:
        doc.close()
    return results


def df_to_records_text(df: pd.DataFrame) -> str:
    if df.empty:
        return ""

    lines = []
    for i, row in df.iterrows():
        fields = ", ".join(f"{col}: {row[col]}" for col in df.columns)
        lines.append(f"Row {i + 1}: {fields}")
    return "\n".join(lines)


def parse_pdf(
    pdf_path: str,
    extract_images: bool = True,
    images_dir: str | None = None,
    doc_id: str | None = None,
) -> dict[str, Any]:
    """Parse a PDF and return a structure with per-page text, tables and images.

    Table text is excluded from page text by removing table bounding boxes
    prior to text extraction.
    """
    pages: list[dict[str, Any]] = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_idx, page in enumerate(pdf.pages, start=1):

            # --- 1. Detect tables with bounding boxes ---
            tables_found = page.find_tables()

            # --- 2. Extract text outside table regions ---
            text_page = page
            for table in tables_found:
                text_page = text_page.outside_bbox(table.bbox)

            text = text_page.extract_text() or ""

            # --- 3. Extract tables from detected tables ---
            tables: list[pd.DataFrame] = []
            for table in tables_found:
                try:
                    df = pd.DataFrame(table.extract())
                except Exception:
                    df = pd.DataFrame()
                tables.append(df)

            # --- 4. Extract images ---
            images: list[dict[str, Any]] = []
            if extract_images and _HAS_FITZ:
                try:
                    images = _extract_images_with_fitz(pdf_path, page_idx, images_dir)
                except Exception as e:
                    print(doc_id, page_idx, e)
                    images = [{"metadata": img} for img in page.images]
            else:
                images = [{"metadata": img} for img in page.images]

            pages.append(
                {
                    "page_number": page_idx,
                    "doc_id": doc_id,
                    "text": text,
                    "tables": tables,
                    "tables_text": "\n".join([df_to_records_text(df) for df in tables]),
                    "full_text": text + "\n".join([df_to_records_text(df) for df in tables]),
                    "images": images,
                }
            )

    return {
        "file_path": pdf_path,
        "doc_id": doc_id,
        "pages": pages,
    }

# 1. Data preparation: 

### 1.1 Loading the pdf files: 

In [3]:
# Getting all files:
import glob
import sys
from tqdm import tqdm

sys.path.append("..")
DATA_PATH = "data/*/*.pdf"
IMAGES_PATH = "data/images/"

data_files = glob.glob(DATA_PATH)
print("Number of files found:", len(data_files))

file_path_to_id = {file_path: file_path.split("/")[-1] for file_path in data_files}

# Parsing all documents: 
all_parsed = []
for fp in tqdm(data_files):
    parsed = parse_pdf(fp, extract_images=True, images_dir=IMAGES_PATH, doc_id=file_path_to_id[fp])
    all_parsed.extend(parsed["pages"])

Number of files found: 41


100%|██████████████████████████████████████████████████████████████████████████████████████████████| 41/41 [00:53<00:00,  1.29s/it]


## 2. Preprocessing documents with LLMs: 

In [4]:
from transformers import pipeline
import torch
from transformers import logging
# Suppress INFO/USER warnings
logging.set_verbosity_error()

pipe = pipeline(
    "image-text-to-text",
    model="google/gemma-3-4b-it",
    device="cuda",
    torch_dtype=torch.bfloat16
)

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█| 883/883 [00:00<00:00, 2646.41it/s, Materializing param=model.vision_tower.vision_model.post_layernorm.weig


### 2.1 Extracting document title: 

How it works: takes the text content from first page of each document and generates the name.

In [5]:
def get_document_title(text):
    PROMPT = """
    Визнач коротку назву документа за текстом першої сторінки.
    Якщо є явний заголовок — додай його, та сформулюй узагальнену назву.
    Поверни лише назву, без пояснень.
    """
    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": PROMPT}]
        },
        {
        "role": "user",
        "content": [
            {"type": "text", "text": text}
        ]
        }
    ]
    output = pipe(text=messages, max_new_tokens=200)
    return output[0]["generated_text"][-1]["content"]

In [6]:
titles_dict = {}
for page in tqdm(all_parsed):
    # Defining the name of the document by the first page: 
    if page["page_number"] == 1:
        titles_dict[page["doc_id"]] = get_document_title(page["text"])

# Assigning the titles to the documents: 
for page in tqdm(all_parsed):
    page["doc_title"] = titles_dict[page["doc_id"]]

100%|█████████████████████████████████████████████████████████████████████████████████████| 1121/1121 [00:00<00:00, 2112225.87it/s]


### 2.2 Extracting images descriptions:

How it works: takes the image, and the text content of the page where the image appears to produce the descriptions.

Note: works pretty long now. Maybe we can do some optimization: shorten/remove the text content or use smaller model. 

In [7]:
from PIL import Image
import base64
import io

def get_images_descriptions(page):
    image_paths = page['images']
    text_content = page['full_text']

    image_descriptions = []
    for image_path in image_paths:
        image = Image.open(image_path["path"])
        # Fix: Convert RGBA → RGB before JPEG encoding
        if image.mode in ('RGBA', 'LA', 'P'):
            image = image.convert('RGB')  # drops alpha
        buffered = io.BytesIO()
        image.save(buffered, format="JPEG")
        img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
        
        PROMPT = """
        Згенеруй короткий опис зображення на сторінці.
        Використовуй наданий візуальний вміст + текст сторінки для контексту.
        Опис має бути інформативний, фактологічний, нейтральний, без фантазій (тільки те, що на картинці).
        Опис повинен чітко відображати всю фактологічну інформацію, щоб можна було відповісти на запитання за картинкою (наприклад текст, всі виміри, ключові деталі). 
        """
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "url": f"data:image/jpeg;base64,{img_b64}"},
                    {"type": "text", "text": f"Текст сторінки: {text_content}"}  # Maybe we can add some kink
                ]
            }
        ]
        output = pipe(text=messages, max_new_tokens=500, do_sample=True)
        image_descriptions.append(output[0]["generated_text"][-1]["content"])
    return "\n".join([f"Рисунок {i}: {text}" for i, text in enumerate(image_descriptions)])

In [ ]:
for page in tqdm(all_parsed):
    if len(page["images"]) > 0:
        page["image_descriptions"] = get_images_descriptions(page)
        page["full_text"] += "\n" + page["image_descriptions"]

 49%|███████████████████████████████████████████▎                                             | 546/1121 [24:24<6:55:41, 43.38s/it]

### 2.4 Dump preprocessed data: 

Note: each document represents the page of the pdf, and include such "main" fields: 
- doc_title
- full_text: text content + parsed content of the tables (in AI readable format) + image descriptions
- other fields (not so important for search)

In [15]:
import joblib
joblib.dump(all_parsed, "data/processed_data.joblib")

['data/processed_data.joblib']

# 3. Building search: 

In [64]:
import joblib
import pandas as pd
import numpy as np
all_parsed = joblib.load("data/processed_data.joblib")

In [65]:
input_df = pd.read_csv("data/dev_questions.csv")
print(len(input_df))
input_df.head(2)

461


,Question_ID,Domain,N_Pages,Question,A,B,C,D,E,F,Correct_Answer,Doc_ID,Page_Num
0,0,domain_2,5,Як рекомендовано приймати ретаболіл дорослим?,внутрішньо,підшкірно,орально,внутрішньовенно,внутрішньом’язово,інгаляційно,E,4e779acee13fa6e0763fb33d1c83030b8e6ea33d.pdf,1
1,1,domain_1,89,Які типи змагань проводяться з боротьби самбо?,тільки індивідуальні,"особисті, командні, особисто-командні",тільки особисті,змагання не проводяться,тільки командні,тільки командні та індивідуальні,B,72e790ea927cb79b08838bcd0e290c83fd4f6e06.pdf,3


### 3.1 Build semantical search (based on embeddings): 
- calculate the embeddings for queries and documents
    - Query is additionally extended with possible answers. 
- select the best ones based on cosine similarity. (no need for ANN, as there are few documents)

In [66]:
import torch
import torch.nn.functional as F
import gc
from tqdm import tqdm

from torch import Tensor
from transformers import AutoTokenizer, AutoModel

def last_token_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]


def get_detailed_instruct(task_description: str, query: str, answers=[]) -> str:
    if len(answers) == 0:
        return f'Instruct: {task_description}\nQuery:{query}'
    else:
        return f'Instruct: {task_description}\nQuery:{query}\nPossible answers:{','.join(answers)}'

max_length = 8192
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-0.6B', padding_side='left')
# model = AutoModel.from_pretrained('Qwen/Qwen3-Embedding-0.6B')
model = AutoModel.from_pretrained('Qwen/Qwen3-Embedding-0.6B', attn_implementation="flash_attention_2", torch_dtype=torch.float16).cuda()

# Each query must come with a one-sentence instruction that describes the task
task = '''
За даним пошуковим запитом (Query), знайди релевантні документи (Document), що містять інформацію щоб відповісти на запитання.
'''

Loading weights: 100%|████████████████████████████████████████| 310/310 [00:00<00:00, 1291.83it/s, Materializing param=norm.weight]


In [67]:
queries = [
    get_detailed_instruct(task, question, answers=answers) for question, answers in zip(input_df["Question"].values, input_df[["A", "B", "C", "D", "E", "F"]].values)
]
# No need to add instruction for retrieval documents
documents = [f"Назва документу: {page['doc_title']}\n Контент: {page['full_text']}" for page in all_parsed]
input_texts = queries + documents

@torch.no_grad()
def embed_texts(
    texts,
    tokenizer,
    model,
    max_length,
    batch_size=32,
):
    all_embeddings = []

    for start_idx in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start_idx : start_idx + batch_size]
        batch_dict = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        batch_dict = {k: v.to(model.device) for k, v in batch_dict.items()}

        outputs = model(**batch_dict)

        embeddings = last_token_pool(
            outputs.last_hidden_state,
            batch_dict["attention_mask"],
        )

        embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.extend(list(embeddings.cpu().numpy()))

        del batch_dict, outputs, embeddings
        gc.collect()
        torch.cuda.empty_cache()
    return all_embeddings

In [68]:
embeddings = embed_texts(
    texts=input_texts,
    tokenizer=tokenizer,
    model=model,
    max_length=max_length,
    batch_size=8,
)

100%|████████████████████████████████████████████████████████████████████████████████████████████| 198/198 [02:48<00:00,  1.17it/s]


### 3.2 Search logic implementation: 

In [69]:
queries_embeddings = np.array(embeddings[:len(queries)])
documents_embeddings = np.array(embeddings[len(queries):])

In [70]:
K = 32 # just to align with future batch sizes (for efficiency)

# Similarity matrix: (num_queries, num_documents)
similarity_matrix = queries_embeddings @ documents_embeddings.T

# Get top-k indices per query (unsorted)
topk_doc_ids = np.argpartition(-similarity_matrix, K, axis=1)[:, :K]

# Optional: sort top-k by similarity score (descending)
topk_scores = np.take_along_axis(similarity_matrix, topk_doc_ids, axis=1)
sorted_idx = np.argsort(-topk_scores, axis=1)

topk_doc_ids = np.take_along_axis(topk_doc_ids, sorted_idx, axis=1)
topk_scores = np.take_along_axis(topk_scores, sorted_idx, axis=1)

In [71]:
retrieved_docs = [
    [all_parsed[i] for i in result]
    for result in topk_doc_ids
]
input_df["retrieved_docs"] = retrieved_docs

### 3.3 Reranking logic: 
Just use the cross-encoder here. Note: it takes time, so we might need to reduce the number of candidates for effitiency. 

**Important comment:** I am dumping the ranking score from cross-encoder. We might use it for post-filtering. (e.g., if the score is <0.01 -> obvious non match). It can help us to reduce the number of candidates for few-shot. 

In [72]:
# Requires transformers>=4.51.0
import torch
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Reranker-0.6B", padding_side='left')
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-Reranker-0.6B", torch_dtype=torch.float16, attn_implementation="flash_attention_2").cuda().eval()
token_false_id = tokenizer.convert_tokens_to_ids("no")
token_true_id = tokenizer.convert_tokens_to_ids("yes")
max_length = 8192

prefix = "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n"
suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
prefix_tokens = tokenizer.encode(prefix, add_special_tokens=False)
suffix_tokens = tokenizer.encode(suffix, add_special_tokens=False)

Loading weights: 100%|██████████████████████████████████| 310/310 [00:00<00:00, 1429.05it/s, Materializing param=model.norm.weight]


In [73]:
def format_instruction(instruction, query, doc):
    if instruction is None:
        instruction = 'Given a web search query, retrieve relevant passages that answer the query'
    output = "<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {doc}".format(instruction=instruction,query=query, doc=doc)
    return output

def process_inputs(pairs):
    inputs = tokenizer(
        pairs, padding=False, truncation='longest_first',
        return_attention_mask=False, max_length=max_length - len(prefix_tokens) - len(suffix_tokens)
    )
    for i, ele in enumerate(inputs['input_ids']):
        inputs['input_ids'][i] = prefix_tokens + ele + suffix_tokens
    inputs = tokenizer.pad(
        inputs, 
        return_tensors="pt", 
        padding=True,
        max_length=max_length,
    )
    for key in inputs:
        inputs[key] = inputs[key].to(model.device)
    return inputs

@torch.no_grad()
def compute_logits(inputs, **kwargs):
    batch_scores = model(**inputs).logits[:, -1, :]
    true_vector = batch_scores[:, token_true_id]
    false_vector = batch_scores[:, token_false_id]
    batch_scores = torch.stack([false_vector, true_vector], dim=1)
    batch_scores = torch.nn.functional.log_softmax(batch_scores, dim=1)
    scores = batch_scores[:, 1].exp().tolist()
    return scores

def get_full_query(query: str, answers=[]) -> str:
    if len(answers) == 0:
        return f'Query:{query}'
    else:
        return f'Query:{query}\nPossible answers:{','.join(answers)}'

# Custom function to fit in memory: 
@torch.no_grad()
def compute_logits_batch(pairs, batch_size=8, **kwargs):
    all_scores = []
    for start_idx in range(0, len(pairs), batch_size):
        batch_pairs = pairs[start_idx : start_idx + batch_size]
        inputs = process_inputs(batch_pairs)
        batch_scores = model(**inputs).logits[:, -1, :]
        true_vector = batch_scores[:, token_true_id]
        false_vector = batch_scores[:, token_false_id]
        batch_scores = torch.stack([false_vector, true_vector], dim=1)
        batch_scores = torch.nn.functional.log_softmax(batch_scores, dim=1)
        scores = batch_scores[:, 1].exp().tolist()
        all_scores.extend(scores)

        del batch_pairs, inputs, batch_scores, true_vector, false_vector, scores
        gc.collect()
        torch.cuda.empty_cache()
    return all_scores

In [74]:
def rerank_documents(question, answers, retrieved_docs):

    task = 'За даним пошуковим запитом (Query), знайди релевантні документи (Document), що містять інформацію щоб відповісти на запитання.'
    documents = [
        f"Назва документу: {page['doc_title']}\n Контент: {page['full_text']}" for page in retrieved_docs
    ]
    query = get_full_query(question, answers=answers)
    pairs = [format_instruction(task, query, doc) for doc in documents]
    scores = compute_logits_batch(pairs)
    
    # Applying scores to documents:
    for document, score in zip(retrieved_docs, scores):
        document["reranking_score"] = score

    return sorted(retrieved_docs, key=lambda e: e["reranking_score"], reverse=True)

In [75]:
from copy import deepcopy
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

reranked_retrieved_docs = []
for _, row in tqdm(input_df.iterrows(), total = len(input_df)):
    
    question_tmp = deepcopy(row["Question"])
    answers_tmp = deepcopy(row[["A", "B", "C", "D", "E", "F"]].values)
    retrieved_docs_tmp = deepcopy(row["retrieved_docs"])
    
    reranked_retrieved_docs.append(rerank_documents(
        question=question_tmp, 
        answers=answers_tmp, 
        retrieved_docs=retrieved_docs_tmp
    ))

input_df["reranked_retrieved_docs"] = reranked_retrieved_docs

100%|████████████████████████████████████████████████████████████████████████████████████████████| 461/461 [45:49<00:00,  5.96s/it]


### 3.4 Search evaluation: 

In [76]:
def recall_at_k(results, ground_truth, k):
    """Compute Recall@K for a single query."""
    retrieved_ids = [doc["doc_id"] for doc in results[:k]]
    return int(ground_truth in retrieved_ids)

def page_score_np(page_pred, page_true, n_pages, d):
    """Compute the page score using numpy."""
    score = (1 - np.abs(page_pred - page_true) / n_pages) * d
    return score

def calculate_metrics_df(df, column="retrieved_docs"):
    # Calculating metrics:
    print("Document search scores: ")
    k_values = [1, 3, 5, 10]
    for k in k_values:
        df[f"recall_at_{k}"] = df.apply(lambda row: recall_at_k(row[column], row["Doc_ID"], k), axis=1)
        print(f"Recall@{k}: {df[f'recall_at_{k}'].mean():.4f}")
    
    # Page score calculation:
    df["correct_doc"] = df.apply(
        lambda row: 1 if row[column][0]["doc_id"] == row["Doc_ID"] else 0, axis=1
    )
    df["page_score"] = df.apply(
        lambda row: page_score_np(
            row[column][0]["page_number"],
            row["Page_Num"],
            row["N_Pages"],
            row["correct_doc"],
        ),
        axis=1,
    )
    print(f"Average Page Score: {df['page_score'].mean():.4f}")

In [77]:
calculate_metrics_df(input_df, column="retrieved_docs")

Document search scores: 
Recall@1: 0.9718
Recall@3: 0.9978
Recall@5: 0.9978
Recall@10: 1.0000
Average Page Score: 0.8668


In [78]:
calculate_metrics_df(input_df, column="reranked_retrieved_docs")

Document search scores: 
Recall@1: 0.9892
Recall@3: 0.9978
Recall@5: 1.0000
Recall@10: 1.0000
Average Page Score: 0.9405


In [87]:
# dump full input_df, that can be used for further modeling: 
joblib.dump(input_df, "data/dev_questions_with_search_results.joblib")

['data/dev_questions_with_search_results.joblib']

### Cells used for errors analysis: 

In [84]:
# for page in all_parsed:
#     if page["doc_id"] == "556be4e553eab6942598e58cfa02b6aaa553a0e5.pdf" and page["page_number"] == 3:
#         print(page)

In [85]:
# pd.set_option('display.max_colwidth', None)
# # retrieved_docs_simplified = [
# #     [{k:v for k, v in all_parsed[i].items() if k in ['page_number', 'doc_id']} for i in result]
# #     for result in topk_doc_ids
# # ]
# retrieved_docs_simplified = [
#     [{k:v for k, v in i.items() if k in ['page_number', 'doc_id', "reranking_score"]} for i in result]
#     for result in input_df["reranked_retrieved_docs"].values
# ]
# input_df["retrieved_docs_simplified"] = retrieved_docs_simplified
# display(input_df[input_df.recall_at_1 != 1][["Question", "Doc_ID", "Page_Num", "retrieved_docs_simplified", "page_score"]])


# pd.set_option('display.max_colwidth', 100)

In [86]:
# pd.set_option('display.max_colwidth', None)
# # retrieved_docs_simplified = [
# #     [{k:v for k, v in all_parsed[i].items() if k in ['page_number', 'doc_id']} for i in result]
# #     for result in topk_doc_ids
# # ]
# retrieved_docs_simplified = [
#     [{k:v for k, v in i.items() if k in ['page_number', 'doc_id', "reranking_score"]} for i in result]
#     for result in input_df["reranked_retrieved_docs"].values
# ]
# input_df["retrieved_docs_simplified"] = retrieved_docs_simplified
# display(input_df[input_df.page_score < 1][["Question", "Doc_ID", "Page_Num", "retrieved_docs_simplified", "page_score"]])


# pd.set_option('display.max_colwidth', 100)